In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm
import math

In [2]:

# ============================
# Configs
# ============================
PARQUET_PATH = "D:\LPA_MTech_Project\Enriched_Datasets\SupremeCourt_Combined_1990_2025_enriched.parquet"      
TEXT_COLUMN = "text"
MODEL_NAME = "google/bert_uncased_L-8_H-512_A-8"
OUTPUT_DIR = "D:\LPA_MTech_Project\My_Models\DAPT\dapt_medium_bert"
MAX_LEN = 128
BATCH_SIZE = 4
ACCUM_STEPS = 16    # Effective batch = 64
LR = 3e-5
EPOCHS = 2

device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
# ============================
# Load parquet
# ============================
print("Loading parquet...")
df = pd.read_parquet(PARQUET_PATH)


Loading parquet...


In [4]:
# Keep only non-empty texts
texts = df[TEXT_COLUMN].dropna().astype(str).tolist()
print(f"Total documents loaded: {len(texts)}")

Total documents loaded: 26142


In [5]:

# ============================
# Dataset
# ============================
class MLMDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        # Remove the extra dimension
        return {k: v.squeeze(0) for k, v in enc.items()}

   

In [6]:

# ============================
# Tokenizer & Model
# ============================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
model.gradient_checkpointing_enable()
model.to(device)

config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/167M [00:00<?, ?B/s]

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 512, padding_idx=0)
      (position_embeddings): Embedding(512, 512)
      (token_type_embeddings): Embedding(2, 512)
      (LayerNorm): LayerNorm((512,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-7): 8 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=512, out_features=512, bias=True)
              (key): Linear(in_features=512, out_features=512, bias=True)
              (value): Linear(in_features=512, out_features=512, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=512, out_features=512, bias=True)
              (LayerNorm): LayerNorm((512,), eps=1e-12, elementwise

In [7]:
# ============================
# DataLoader
# ============================
dataset = MLMDataset(texts, tokenizer, MAX_LEN)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
)

In [8]:
# ============================
# Optimizer & Scheduler
# ============================
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

num_training_steps = len(loader) * EPOCHS // ACCUM_STEPS
warmup_steps = int(0.1 * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    warmup_steps,
    num_training_steps
)

scaler = torch.cuda.amp.GradScaler()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16880\3748111210.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [9]:

# ============================
# Training Loop
# ============================
model.train()
step = 0

print("\nStarting DAPT/MLM Training...\n")

for epoch in range(EPOCHS):
    epoch_loss = 0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    optimizer.zero_grad()

    for batch in pbar:
        # batch = {k: v.squeeze().to(device) for k, v in batch.items()}
        batch = {k: v.to(device) for k, v in batch.items()}


        with torch.cuda.amp.autocast():
            outputs = model(**batch)
            loss = outputs.loss / ACCUM_STEPS

        scaler.scale(loss).backward()
        epoch_loss += loss.item() * ACCUM_STEPS

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        step += 1
        pbar.set_postfix({"loss": epoch_loss / (step+1)})

    ppl = math.exp(epoch_loss / len(loader))
    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(loader):.4f} | Perplexity: {ppl:.2f}")

    model.save_pretrained(f"{OUTPUT_DIR}/epoch_{epoch+1}")
    tokenizer.save_pretrained(f"{OUTPUT_DIR}/epoch_{epoch+1}")



Starting DAPT/MLM Training...



Epoch 1/2:   0%|          | 0/6536 [00:00<?, ?it/s]

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16880\3524395809.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1 Loss: 2.4798 | Perplexity: 11.94


Epoch 2/2:   0%|          | 0/6536 [00:00<?, ?it/s]

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16880\3524395809.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 2 Loss: 2.0783 | Perplexity: 7.99


In [10]:

print("\nTraining Complete!")
model.save_pretrained(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")
print("Saved DAPT model successfully.")


Training Complete!
Saved DAPT model successfully.
